# Functional API examples

See the [Functional API reference](reference.rst) for the list of available functions.


The functional API (`dtv_fn`) exposes stateless functions — the same operations without
needing operator objects or a `Compose` wrapper.
The call signature intentionally mirrors `torchvision.transforms.v2.functional`.

```python
import nvidia.dali.experimental.torchvision.v2.functional as dtv_fn

output = dtv_fn.resize(img, size=(224, 224))
```

Both `PIL.Image` and `torch.Tensor` inputs are supported for most functions. A tensor is either a single `CHW` image (3-D) or a batch of images `(N, C, H, W)` (4-D). Tensors with more leading dimensions, e.g. videos, are supported by `Compose` only.

## Setup

Import the DALI TorchVision API (object-oriented as `dtv`, functional as `dtv_fn`) alongside the standard TorchVision counterparts for comparison.

In [1]:
import torch
import torchvision.transforms.v2 as tv
import torchvision.transforms.v2.functional as tv_fn
from PIL import Image
import numpy as np

# DALI TorchVision API — object-oriented
import nvidia.dali.experimental.torchvision as dtv

# DALI TorchVision API — functional
import nvidia.dali.experimental.torchvision.v2.functional as dtv_fn

print("DALI operators available:", dtv.__all__)
print("DALI functional ops available:", dtv_fn.__all__)

DALI operators available: ['CenterCrop', 'ColorJitter', 'Compose', 'GaussianBlur', 'Grayscale', 'InterpolationMode', 'Normalize', 'Pad', 'PILToTensor', 'RandomApply', 'RandomCrop', 'RandomGrayscale', 'RandomHorizontalFlip', 'RandomResizedCrop', 'RandomVerticalFlip', 'Resize', 'ToPILImage', 'ToPureTensor']
DALI functional ops available: ['center_crop', 'crop', 'gaussian_blur', 'get_dimensions', 'get_image_size', 'get_size', 'horizontal_flip', 'normalize', 'pad', 'pil_to_tensor', 'resize', 'resized_crop', 'rgb_to_grayscale', 'to_grayscale', 'to_pil_image', 'to_tensor', 'vertical_flip']


In [2]:
# A sample image, as a CHW uint8 tensor and as a PIL Image
ramp = torch.linspace(0, 255, 64)
tensor_img = torch.stack(
    [
        ramp.expand(64, 64),
        ramp[:, None].expand(64, 64),
        (ramp + ramp[:, None]) / 2,
    ]
).to(torch.uint8)
pil_img = Image.fromarray(tensor_img.permute(1, 2, 0).numpy())

In [3]:
# Functional API on a PIL Image
resized_pil = dtv_fn.resize(pil_img, size=[32, 32])
flipped_pil = dtv_fn.horizontal_flip(pil_img)
gray_pil = dtv_fn.to_grayscale(pil_img)
padded_pil = dtv_fn.pad(pil_img, padding=8, padding_mode="edge")
cropped_pil = dtv_fn.center_crop(pil_img, output_size=(48, 48))

print("resize       :", resized_pil.size)
print("hflip        :", flipped_pil.size)
print("grayscale    :", gray_pil.size, gray_pil.mode)
print("pad (edge, 8):", padded_pil.size)
print("center_crop  :", cropped_pil.size)

resize       : (32, 32)
hflip        : (64, 64)
grayscale    : (64, 64) L
pad (edge, 8): (80, 80)
center_crop  : (48, 48)


In [4]:
# Functional API on a torch.Tensor (CHW, float32)
# Scale tensor_img to [0, 1] dynamic range before processing
scaled_tensor_img = tensor_img / tensor_img.max()

resized_t = dtv_fn.resize(scaled_tensor_img, size=[32, 32])
flipped_t = dtv_fn.horizontal_flip(scaled_tensor_img)
norm_t = dtv_fn.normalize(
    scaled_tensor_img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
)
blurred_t = dtv_fn.gaussian_blur(scaled_tensor_img, kernel_size=5, sigma=1.0)

print("resize       :", resized_t.shape)
print("hflip        :", flipped_t.shape)
print("normalize    :", norm_t.shape, "mean:", norm_t.mean().item())
print("gaussian_blur:", blurred_t.shape)

resize       : torch.Size([3, 32, 32])
hflip        : torch.Size([3, 64, 64])
normalize    : torch.Size([3, 64, 64]) mean: 0.21830737590789795
gaussian_blur: torch.Size([3, 64, 64])
